# PoolPy Pool Design - User-Friendly Interface

This notebook provides an easy-to-use interface for generating well assignment (WA) matrices using the PoolPy pool design generator.

## Instructions:
1. **Cell 1**: Verify environment and dependencies
2. **Cell 2**: Import required libraries
3. **Cell 3**: Define your pool design configuration (parameters)
4. **Cell 4**: Run the pool generator
5. **Cell 5**: View and verify generated WA matrices

Simply fill in the configuration variables in Cell 3 and run the cells in order.


In [1]:
import subprocess
import sys
import os

# Setup environment from requirements.txt
print("Environment Setup")
print("=" * 70)

# Get the working directory
working_dir = os.getcwd()
print(f"Working directory: {working_dir}")
print()

requirements_path = os.path.join(working_dir, "requirements.txt")

# Try to read requirements.txt to show what packages are needed
try:
    with open(requirements_path, 'r') as f:
        requirements = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    
    print(f"✓ Requirements file found: {requirements_path}")
    print(f"\nRequired packages ({len(requirements)} total):")
    for req in requirements[:10]:  # Show first 10
        print(f"  - {req}")
    if len(requirements) > 10:
        print(f"  ... and {len(requirements) - 10} more")
    
    print("\n" + "-" * 70)
    print("Environment Status:")
    print(f"  Python version: {sys.version}")
    print(f"  Python executable: {sys.executable}")
    
    # Check if key packages are available
    key_packages = ['pandas', 'numpy', 'scipy', 'sklearn']
    for pkg_name in key_packages:
        try:
            __import__(pkg_name)
            print(f"  ✓ {pkg_name} is installed")
        except ImportError:
            print(f"  ✗ {pkg_name} is NOT installed")
    
except FileNotFoundError:
    print(f"✗ Requirements file not found: {requirements_path}")
    print("Please ensure requirements.txt exists in the working directory.")

print("=" * 70)
print("\nNote: For UV environments, dependencies are typically pre-installed.")
print("If you need to install packages, use: uv pip install -r requirements.txt")


Environment Setup
Working directory: /Users/ltalamanca/My Drive/Git/PoolPy

✓ Requirements file found: /Users/ltalamanca/My Drive/Git/PoolPy/requirements.txt

Required packages (50 total):
  - appnope==0.1.4
  - asttokens==3.0.0
  - cmcrameri==1.9
  - comm==0.2.2
  - contourpy==1.3.2
  - cycler==0.12.1
  - debugpy==1.8.14
  - decorator==5.2.1
  - et-xmlfile==2.0.0
  - executing==2.2.0
  ... and 40 more

----------------------------------------------------------------------
Environment Status:
  Python version: 3.12.9 (main, Mar 17 2025, 21:36:21) [Clang 20.1.0 ]
  Python executable: /Users/ltalamanca/uv_2/bin/python
  ✓ pandas is installed
  ✓ numpy is installed
  ✓ scipy is installed
  ✓ sklearn is installed

Note: For UV environments, dependencies are typically pre-installed.
If you need to install packages, use: uv pip install -r requirements.txt


In [2]:
import pandas as pd
import numpy as np
import subprocess
import os
from pathlib import Path

print("✓ All libraries imported successfully!")
print(f"\nPython executable: {sys.executable}")
print(f"Working directory: {os.getcwd()}")


✓ All libraries imported successfully!

Python executable: /Users/ltalamanca/uv_2/bin/python
Working directory: /Users/ltalamanca/My Drive/Git/PoolPy


## Pool Design Configuration

Edit the variables below to configure your well assignment (WA) matrix generation.

**Mode Selection:**
- **Range Mode** (default): Generate WA matrices for multiple pool counts. Set `n_samp=0` and use `start`, `stop`, `step`
- **Single Sample Mode**: Generate a WA matrix for a specific number of pools. Set `n_samp` to your desired pool count

**Variables:**
- `start`: Starting number of samples (range mode) - default: 50
- `stop`: Ending number of samples (range mode) - default: 110
- `step`: Step size between sample counts (range mode) - default: 10
- `n_samp`: For single sample mode, set to desired pool count. 0 = use range mode
- `directory`: Output directory for generated WA matrices - default: './pooling_designs'
- `max_diff`: Maximum number of positive samples value to consider - default: 10 (the code will also generate designs for all number of posistives from 1 to max_diff)
- `max_redundancy`: Maximum redundancy for random designs - default: 2.0
- `min_redundancy`: Minimum redundancy for random designs - default: 0.5
- `max_prev`: Other bound to the maximum nuber of positives, this time described as a fraction of the number of samples - default: 0.1
- `max_dims`: Maximum dimensions to consider - default: infinity
- `rand_guesses`: Number of random guesses to try - default: 10
- `one_liner`: Output in single line format (True/False) - default: True
- `cleanup`: Remove intermediate files (True/False) - default: False
- `overwrite`: Overwrite existing files (True/False) - default: True
- `timeit`: Measure execution time (True/False) - default: True


In [3]:
# ============================================================================
# USER CONFIGURATION - EDIT THESE VARIABLES
# ============================================================================

# Get the working directory (set automatically)
working_dir = os.getcwd()

# ============================================================================
# CORE PARAMETERS
# ============================================================================

# OUTPUT DIRECTORY - where to save generated WA matrices
output_directory = os.path.join(working_dir, "pooling_designs")

# ============================================================================
# CHOOSE ONE MODE: RANGE MODE or SINGLE SAMPLE MODE
# ============================================================================

# MODE 1: RANGE MODE (generate for multiple pool counts)
# Set n_samp = 0 to enable range mode
n_samp = 0  # Set to 0 for range mode

# When using RANGE MODE, these parameters control the pool count range:
start = 20      # Starting number of pools
stop = 30      # Ending number of pools
step = 10       # Step size (will generate matrices for 50, 60, 70, 80, 90, 100, 110 pools)

# MODE 2: SINGLE SAMPLE MODE (generate for one specific pool count)
# Uncomment below and set n_samp to the desired number of pools
# n_samp = 80   # Generates WA matrix for 80 pools only

# ============================================================================
# OPTIMIZATION PARAMETERS
# ============================================================================

max_diff = 10                # Maximum number of positive samples value to consider 
max_redundancy = 2.0         # Maximum redundancy for random design
min_redundancy = 0.5         # Minimum redundancy for random design
max_prev = 0.1               # Other bound to the maximum nuber of positives, this time described as a fraction of the number of samples
max_dims = np.inf            # Maximum dimensions to consider (use np.inf for unlimited)

# Number of random guesses to try when optimizing
rand_guesses = 10

# ============================================================================
# OUTPUT OPTIONS
# ============================================================================

one_liner = True             # Output in single line format (True/False)
cleanup = False              # Remove intermediate files (True/False)
overwrite = True             # Overwrite existing files (True/False)
timeit = True                # Measure execution time (True/False)

# ============================================================================
# VALIDATION AND DISPLAY
# ============================================================================

print("Pool Design Configuration")
print("=" * 70)
print(f"Working Directory: {working_dir}")
print(f"Output Directory:  {os.path.basename(output_directory)}")
print()

# Display active mode
if n_samp > 0:
    print(f"MODE: SINGLE SAMPLE")
    print(f"  Generating WA matrix for {n_samp} pools")
else:
    print(f"MODE: RANGE")
    print(f"  Pool count range: {start} to {stop} (step: {step})")
    pool_counts = list(range(start, stop + 1, step))
    print(f"  Will generate matrices for: {pool_counts}")

print()
print("Core Parameters:")
print(f"  Max Diff:         {max_diff}")
print(f"  Max Redundancy:   {max_redundancy}")
print(f"  Min Redundancy:   {min_redundancy}")
print(f"  Max Prev:         {max_prev}")
print(f"  Rand Guesses:     {rand_guesses}")
print()
print("Execution Options:")
print(f"  One Liner:        {one_liner}")
print(f"  Cleanup:          {cleanup}")
print(f"  Overwrite:        {overwrite}")
print(f"  Timeit:           {timeit}")
print("=" * 70)

# Create output directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
    print(f"\n✓ Created output directory: {output_directory}")
else:
    print(f"\n✓ Output directory exists: {output_directory}")


Pool Design Configuration
Working Directory: /Users/ltalamanca/My Drive/Git/PoolPy
Output Directory:  pooling_designs

MODE: RANGE
  Pool count range: 20 to 30 (step: 10)
  Will generate matrices for: [20, 30]

Core Parameters:
  Max Diff:         10
  Max Redundancy:   2.0
  Min Redundancy:   0.5
  Max Prev:         0.1
  Rand Guesses:     10

Execution Options:
  One Liner:        True
  Cleanup:          False
  Overwrite:        True
  Timeit:           True

✓ Created output directory: /Users/ltalamanca/My Drive/Git/PoolPy/pooling_designs


## Execute Pool Generator

Run pool_N.py with the configuration settings from the previous cell.


In [4]:
# Use the current Python executable
exec_python = sys.executable

# Path to pool_N.py script
script_path = os.path.join(working_dir, "pool_N.py")

if not os.path.exists(script_path):
    print(f"Error: pool_N.py not found at {script_path}")
    print(f"Expected location: {working_dir}")
else:
    # Build the pool generator command
    cmd = [
        exec_python,
        script_path,
        "--directory", output_directory,
        "--max_diff", str(max_diff),
        "--max_redundancy", str(max_redundancy),
        "--min_redundancy", str(min_redundancy),
        "--max_prev", str(max_prev),
        "--rand_guesses", str(rand_guesses),
        "--one_liner", "True" if one_liner else "False",
        "--cleanup", "True" if cleanup else "False",
        "--overwrite", "True" if overwrite else "False",
        "--timeit", "True" if timeit else "False",
        "--n_samp", str(n_samp)
    ]
    
    # Add range parameters only if not in single sample mode
    if n_samp == 0:
        cmd.extend(["--start", str(start)])
        cmd.extend(["--stop", str(stop)])
        cmd.extend(["--step", str(step)])
    
    # Add max_dims if not infinity
    if not np.isinf(max_dims):
        cmd.extend(["--max_dims", str(int(max_dims))])
    
    # Display the command being executed
    print("Executing pooling design generator...")
    print("=" * 70)
    print(f"Command: {' '.join(cmd)}")
    print("=" * 70)
    print()
    
    # Run the pool generator
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            cwd=working_dir,
            timeout=3600  # 1-hour timeout
        )
        
        print(f"Return code: {result.returncode}")
        print()
        
        if result.stdout:
            print("STDOUT:")
            print(result.stdout)
        
        if result.stderr:
            print("STDERR:")
            print(result.stderr)
        
        # Check for success
        if result.returncode == 0:
            print("\n" + "=" * 70)
            print("✓ POOL GENERATION COMPLETED SUCCESSFULLY")
            print("=" * 70)
        else:
            print("\n" + "=" * 70)
            print("✗ POOL GENERATION FAILED")
            print("=" * 70)
    
    except subprocess.TimeoutExpired:
        print("✗ Pool generator execution timed out (1 hour)")
    except Exception as e:
        print(f"✗ Error running pool generator: {e}")


Executing pooling design generator...
Command: /Users/ltalamanca/uv_2/bin/python /Users/ltalamanca/My Drive/Git/PoolPy/pool_N.py --directory /Users/ltalamanca/My Drive/Git/PoolPy/pooling_designs --max_diff 10 --max_redundancy 2.0 --min_redundancy 0.5 --max_prev 0.1 --rand_guesses 10 --one_liner True --cleanup False --overwrite True --timeit True --n_samp 0 --start 20 --stop 30 --step 10

Return code: 0

STDOUT:
Processing 20 compounds
Saved multidim-3 for diff=1
Saved Matrix for diff=1
Saved Binary for diff=1
Computed STD for diff=1
Computed Chinese remainder for diff=1
Computed Ch. rm. bktrk for diff=1
Copied multidim-3 to diff=2
Copied Matrix to diff=2
Copied Binary to diff=2
Computed STD for diff=2
Computed Chinese remainder for diff=2
Computed Ch. rm. bktrk for diff=2
Computed Chinese special for diff=2


0.0 days 0.0 hours 0.0 minutes and 0.01 seconds required for N= 20 compounds


Processing 30 compounds
Saved multidim-3 for diff=1
Saved multidim-4 for diff=1
Saved Matrix for dif

## View Generated WA Matrices

Display and verify the generated well assignment matrices.


In [7]:
# List and display generated WA matrices
print("Generated WA Matrices")
print("=" * 70)

if not os.path.exists(output_directory):
    print(f"✗ Output directory not found: {output_directory}")
else:
    # Recursively find all CSV files (pool_N.py stores them in N_*/diff_*/WAs/)
    csv_files = []
    for root, dirs, files in os.walk(output_directory):
        dirs.sort()
        for fname in sorted(files):
            if fname.endswith('.csv'):
                csv_files.append(os.path.join(root, fname))

    if not csv_files:
        print(f"⚠ No CSV files found under {output_directory}")
        print("The pool generator may not have created any matrices yet.")
    else:
        print(f"\n✓ Found {len(csv_files)} WA matrix file(s):\n")

        for i, csv_path in enumerate(csv_files, 1):
            rel_path = os.path.relpath(csv_path, output_directory)
            try:
                df = pd.read_csv(csv_path, index_col=0)
                n_compounds = len(df)
                n_pools = len(df.columns)
                print(f"{i}. {rel_path}")
                print(f"   - Compounds: {n_compounds}  |  Pools: {n_pools}  |  Shape: {df.shape}")
                preview = df.iloc[:3, :10]
                for line in preview.to_string().split('\n'):
                    print(f"     {line}")
                print()
            except Exception as e:
                print(f"{i}. {rel_path} - Error reading file: {e}\n")

        print("=" * 70)
        print(f"✓ All WA matrices saved under: {output_directory}")
        print("=" * 70)

Generated WA Matrices

✓ Found 31 WA matrix file(s):

1. N_20/diff_1/WAs/WA_Binary_N_20_diff_1.csv
   - Compounds: 20  |  Pools: 5  |  Shape: (20, 5)
               Pool_0  Pool_1  Pool_2  Pool_3  Pool_4
     Sample_0       0       0       0       0       1
     Sample_1       0       0       0       1       0
     Sample_2       0       0       0       1       1

2. N_20/diff_1/WAs/WA_Ch. rm. bktrk_N_20_diff_1.csv
   - Compounds: 20  |  Pools: 9  |  Shape: (20, 9)
               Pool_0  Pool_1  Pool_2  Pool_3  Pool_4  Pool_5  Pool_6  Pool_7  Pool_8
     Sample_0       1       0       0       0       1       0       0       0       0
     Sample_1       0       1       0       0       0       1       0       0       0
     Sample_2       0       0       1       0       0       0       1       0       0

3. N_20/diff_1/WAs/WA_Matrix_N_20_diff_1.csv
   - Compounds: 20  |  Pools: 9  |  Shape: (20, 9)
               Pool_0  Pool_1  Pool_2  Pool_3  Pool_4  Pool_5  Pool_6  Pool_7  Pool_8
   